# Spin Chains & the NPA Hierarchy

This notebook walks through the complete **spin-chain** pipeline, from exact diagonalisation to SDP relaxations and basis optimisation.

| Step | What you'll see |
|------|----------------|
| **1** | Setup & imports |
| **2** | Spin-chain Hamiltonians (Heisenberg model) |
| **3** | Exact ground-state energy with QuTiP |
| **4** | Pauli-word algebra |
| **5** | NPA basis generation for spins |
| **6** | Moment-matrix compilation |
| **7** | Solving the SDP relaxation (NPA lower bound) |
| **8** | Comparing NPA levels to exact values |
| **9** | Symmetries: same bounds, smaller SDPs |
| **10** | Basis-selection optimisation |
| **11** | Sweeping over *k* values and seeds |
| **12** | Atomic persistence with a callback |
| **13** | Loading and discovering saved results |

---
## 1. Setup & Imports

In [22]:
import sys, json, shutil, time
import numpy as np
from pathlib import Path

# Add the project root so top-level packages are importable
PROJECT_ROOT = str(Path("..").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- Spin models (exact + Pauli-dict forms) ---
from spins.models import (
    heisenberg_hamiltonian_exact,
    heisenberg_hamiltonian_dict,
    ising_hamiltonian_exact,
    ising_hamiltonian_dict,
)

# --- Pauli algebra ---
from spins.pauli_logic import (
    PauliWord,
    multiply_words,
    local_pauli,
    compile_moment_matrix_rep,
)

# --- NPA basis builder ---
from spins.basis_builder import (
    generate_npa_basis,
    generate_heisenberg_paper_basis,
)

# --- Symmetries ---
from spins.symmetry import SymmetryManager

# --- SDP layer ---
from spins.spins_sdp import (
    build_block_reps,
    build_block_diagonal_sdp,
    compile_operator_linear_form,
    solve_pauli_relaxation,
)

# --- Optimisation layer ---
from spins.spins_optimize import (
    build_npa_basis_sets,
    optimize_ground_energy,
    sweep_k_values,
    SpinOptimizationResult,
    OnResultCallback,
)

# --- Persistence ---
from artifact_manager import ArtifactManager, RunDir, config_hash

print("All imports OK ✓")
print(f"Project root: {PROJECT_ROOT}")

All imports OK ✓
Project root: /users/eleves-b/2023/sherab-losel.matos/thesis/SpinsSDP/SpinsSDP


---
## 2. Spin-Chain Hamiltonians

We focus on the **antiferromagnetic Heisenberg chain** with periodic boundary conditions (PBC):

$$H = \frac{1}{4}\sum_{i=1}^{N}\sum_{a \in \{x,y,z\}} \sigma_i^a \, \sigma_{i+1}^a$$

where $\sigma_i^a$ are Pauli matrices on site $i$ and $N+1 \equiv 1$ (PBC).

The library provides two representations of each Hamiltonian:
- **Exact** (QuTiP `Qobj`): for full diagonalisation on small systems.
- **Pauli dictionary** (`Dict[PauliWord, complex]`): symbolic form used by the SDP layer.

In [23]:
# Build the Heisenberg Hamiltonian for a small chain (N=4)
N = 4

H_exact = heisenberg_hamiltonian_exact(N, boundary="periodic")
H_dict  = heisenberg_hamiltonian_dict(N, boundary="periodic")

print(f"Heisenberg chain: N = {N}, PBC")
print(f"  QuTiP Hamiltonian: {H_exact.shape[0]}×{H_exact.shape[0]} matrix")
print(f"  Pauli-dict terms:  {len(H_dict)} Pauli words\n")

print("Pauli-dictionary terms:")
for word, coeff in sorted(H_dict.items(), key=lambda t: (t[0].support_size(), t[0].x_mask, t[0].z_mask)):
    print(f"  {coeff:+.4f} · {word}")

Heisenberg chain: N = 4, PBC
  QuTiP Hamiltonian: 16×16 matrix
  Pauli-dict terms:  12 Pauli words

Pauli-dictionary terms:
  +0.2500 · Z0Z1
  +0.2500 · Z1Z2
  +0.2500 · Z0Z3
  +0.2500 · Z2Z3
  +0.2500 · X0X1
  +0.2500 · Y0Y1
  +0.2500 · X1X2
  +0.2500 · Y1Y2
  +0.2500 · X0X3
  +0.2500 · Y0Y3
  +0.2500 · X2X3
  +0.2500 · Y2Y3


---
## 3. Exact Ground-State Energy with QuTiP

For small systems ($N \leq 14$ or so) we can find the exact ground-state energy $E_0$ via full diagonalisation. This serves as our reference for benchmarking the SDP relaxation.

In [3]:
# Compute exact ground-state energies for several system sizes
print(f"{'N':>3s}  {'dim':>6s}  {'E0':>12s}  {'E0/N':>12s}  {'time (s)':>10s}")
print("-" * 50)

exact_energies = {}
for N in [4, 6, 8, 10]:
    t0 = time.perf_counter()
    H = heisenberg_hamiltonian_exact(N, boundary="periodic")
    evals = H.eigenenergies(eigvals=1)
    E0 = float(evals[0])
    dt = time.perf_counter() - t0
    exact_energies[N] = E0
    print(f"{N:>3d}  {2**N:>6d}  {E0:>12.6f}  {E0/N:>12.6f}  {dt:>10.4f}")

print(f"\nExact E0 for N=6: {exact_energies[6]:.6f}")

  N     dim            E0          E0/N    time (s)
--------------------------------------------------
  4      16     -2.000000     -0.500000      0.0191
  6      64     -2.802776     -0.467129      0.0062
  8     256     -3.651093     -0.456387      0.0123
 10    1024     -4.515446     -0.451545      0.0884

Exact E0 for N=6: -2.802776


---
## 4. Pauli-Word Algebra

A `PauliWord` encodes a tensor product of Pauli matrices using two bitmasks: `x_mask` and `z_mask`.

| Bit pattern $(x, z)$ | Operator |
|:---------------------:|:--------:|
| $(0, 0)$              | $I$      |
| $(1, 0)$              | $X$      |
| $(1, 1)$              | $Y$      |
| $(0, 1)$              | $Z$      |

The product of two Pauli words gives a phase $i^p$ times another Pauli word:
$$\sigma_A \cdot \sigma_B = i^p \, \sigma_C$$

In [24]:
# Create single-site Pauli operators
X0 = local_pauli(0, "x")
Y0 = local_pauli(0, "y")
Z0 = local_pauli(0, "z")
Z1 = local_pauli(1, "z")
I  = PauliWord(0, 0)

print(f"X0 = {X0}   (x_mask={X0.x_mask:#04b}, z_mask={X0.z_mask:#04b})")
print(f"Y0 = {Y0}   (x_mask={Y0.x_mask:#04b}, z_mask={Y0.z_mask:#04b})")
print(f"Z0 = {Z0}   (x_mask={Z0.x_mask:#04b}, z_mask={Z0.z_mask:#04b})")
print(f"Z1 = {Z1}   (x_mask={Z1.x_mask:#04b}, z_mask={Z1.z_mask:#04b})")
print(f"I  = {I}")

# Multiplication: X0 * Y0 = i * Z0
phase, result = multiply_words(X0, Y0)
print(f"\nX0 · Y0 = i^{phase} · {result}   (i.e. i·Z0)")

# Two-site product: Z0 * Z1 = Z0Z1 (phase 0)
phase2, result2 = multiply_words(Z0, Z1)
print(f"Z0 · Z1 = i^{phase2} · {result2}")

# Pauli words are idempotent: X0 * X0 = I
phase3, result3 = multiply_words(X0, X0)
print(f"X0 · X0 = i^{phase3} · {result3}   (identity)")

X0 = X0   (x_mask=0b01, z_mask=0b00)
Y0 = Y0   (x_mask=0b01, z_mask=0b01)
Z0 = Z0   (x_mask=0b00, z_mask=0b01)
Z1 = Z1   (x_mask=0b00, z_mask=0b10)
I  = I

X0 · Y0 = i^1 · Z0   (i.e. i·Z0)
Z0 · Z1 = i^0 · Z0Z1
X0 · X0 = i^0 · I   (identity)


---
## 5. NPA Basis Generation for Spins

The NPA hierarchy at level $k$ builds a set of Pauli words by taking all products of up to $k$ single-site generators $\{X_i, Y_i, Z_i\}$.

- **Level 0:** $\{I\}$
- **Level 1:** $\{I, X_0, Y_0, Z_0, X_1, Y_1, Z_1, \ldots\}$
- **Level 2:** adds all pairwise products $X_0 X_1$, $X_0 Y_1$, etc.

`generate_npa_basis(N, k)` returns a `PauliNPABasis` with words grouped by level.

In [26]:
# Generate NPA bases for a small chain (N=4)
N = 4

basis_k1 = generate_npa_basis(N, k=1)
basis_k2 = generate_npa_basis(N, k=2)

print(f"=== NPA Level 1 (N={N}) ===")
print(f"  Total words: {len(basis_k1.words)}")
for lvl, words in enumerate(basis_k1.levels):
    names = [repr(w) for w in words[:8]]
    suffix = " ..." if len(words) > 8 else ""
    print(f"  Level {lvl}: {len(words)} words → {names}{suffix}")

print(f"\n=== NPA Level 2 (N={N}) ===")
print(f"  Total words: {len(basis_k2.words)}")
for lvl, words in enumerate(basis_k2.levels):
    print(f"  Level {lvl}: {len(words)} words")

=== NPA Level 1 (N=4) ===
  Total words: 13
  Level 0: 1 words → ['I']
  Level 1: 12 words → ['Z0', 'Z1', 'Z2', 'Z3', 'X0', 'Y0', 'X1', 'Y1'] ...

=== NPA Level 2 (N=4) ===
  Total words: 67
  Level 0: 1 words
  Level 1: 12 words
  Level 2: 54 words


In [27]:
# A paper-specific (arXiv preprint arXiv:2310.05844, 2023) basis for the Heisenberg model
# (includes contiguous triples and quadruples - more physically motivated)
paper_basis = generate_heisenberg_paper_basis(N=4)
print(f"Paper basis for N=4: {len(paper_basis)} words")
print(f"First 20 words: {[repr(w) for w in paper_basis[:20]]}")

Paper basis for N=4: 256 words
First 20 words: ['I', 'X0', 'Y0', 'Z0', 'X1', 'Y1', 'Z1', 'X2', 'Y2', 'Z2', 'X3', 'Y3', 'Z3', 'X0X1', 'X0Y1', 'X0Z1', 'Y0X1', 'Y0Y1', 'Y0Z1', 'Z0X1']


---
## 6. Moment-Matrix Compilation

Given a basis $\{w_0, w_1, \ldots, w_{n-1}\}$, the **moment matrix** $\Gamma$ has entries:

$$\Gamma_{ij} = \langle \psi | w_i^\dagger \, w_j | \psi \rangle$$

Since each $w_i$ is a Pauli word, $w_i^\dagger w_j$ reduces (up to a phase) to another Pauli word $u$. So each entry of $\Gamma$ is $\pm y_u$ or $\pm i \, y_u$ where $y_u = \langle u \rangle$.

`compile_moment_matrix_rep(basis, symmetry_manager)` computes this mapping. The result tells us:
- `labels`: the unique moment variables $y_u$
- `label_idx[i,j]`: which variable appears at position $(i, j)$
- `a_coef[i,j]`, `b_coef[i,j]`: the real/imaginary coefficients

In [28]:
# Compile the moment matrix for NPA level 1, no symmetries
N_demo = 4
sym_none = SymmetryManager.none(N=N_demo)
basis_demo = generate_npa_basis(N_demo, k=1)

rep = compile_moment_matrix_rep(basis_demo.words, sym_none)

print(f"Basis size (moment matrix dimension): {len(basis_demo.words)} × {len(basis_demo.words)}")
print(f"Number of unique moment variables:     {len(rep.labels)}")
print(f"Identity label index:                  {rep.idx_I}")
print(f"\nFirst 10 moment labels:")
for i, label in enumerate(rep.labels[:10]):
    print(f"  y[{i}] = ⟨{label}⟩")

Basis size (moment matrix dimension): 13 × 13
Number of unique moment variables:     67
Identity label index:                  0

First 10 moment labels:
  y[0] = ⟨I⟩
  y[1] = ⟨Z0⟩
  y[2] = ⟨Z1⟩
  y[3] = ⟨Z2⟩
  y[4] = ⟨Z3⟩
  y[5] = ⟨X0⟩
  y[6] = ⟨Y0⟩
  y[7] = ⟨X1⟩
  y[8] = ⟨Y1⟩
  y[9] = ⟨X2⟩


---
## 7. Solving the SDP Relaxation

The SDP relaxation asks: *find values $y_u$ such that the moment matrix $\Gamma \succeq 0$, $\langle I \rangle = 1$, and the energy $\langle H \rangle = \sum_u c_u y_u$ is minimised.*

This gives a **certified lower bound** on the ground-state energy:

$$E_{\text{SDP}} \leq E_0$$

Let's solve this for N=6 at NPA level 1, using no symmetries first.

In [8]:
# Solve the SDP relaxation for N=6, NPA level 1 (no symmetries)
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
sym_none_6 = SymmetryManager.none(N=N)

basis_lv1 = generate_npa_basis(N, k=1)
print(f"N={N}, NPA level 1: {len(basis_lv1.words)} basis words")

t0 = time.perf_counter()
E_sdp = solve_pauli_relaxation(
    basis=basis_lv1.words,
    operator=H_dict_6,
    symmetry_manager=sym_none_6,
    sense="min",
    mosek_tol=1e-7,
)
dt = time.perf_counter() - t0

E_exact_6 = exact_energies[6]
gap = E_exact_6 - E_sdp

print(f"\n{'='*50}")
print(f"  SDP lower bound (NPA lv1): {E_sdp:.6f}")
print(f"  Exact ground energy:       {E_exact_6:.6f}")
print(f"  Gap (E0 - E_SDP):          {gap:.6f}")
print(f"  Relative gap:              {abs(gap / E_exact_6):.4%}")
print(f"  Solve time:                {dt:.3f}s")
print(f"{'='*50}")

N=6, NPA level 1: 19 basis words

  SDP lower bound (NPA lv1): -4.500000
  Exact ground energy:       -2.802776
  Gap (E0 - E_SDP):          1.697224
  Relative gap:              60.5551%
  Solve time:                0.203s


---
## 8. Comparing NPA Levels to Exact Values

Higher NPA levels include more Pauli words → larger moment matrix → tighter bound. Let's see how the gap shrinks.

In [29]:
# Compare NPA levels 1 and 2 against exact energy for N=6
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
E_exact_6 = exact_energies[6]

print(f"Heisenberg chain N={N}, PBC")
print(f"Exact ground-state energy: {E_exact_6:.6f}\n")
print(f"{'Level':>6s}  {'Basis size':>10s}  {'E_SDP':>12s}  {'Gap':>12s}  {'Rel. gap':>10s}  {'Time (s)':>10s}")
print("-" * 70)

sym_none_6 = SymmetryManager.none(N=N)

for k in [1, 2]:
    basis = generate_npa_basis(N, k=k)
    t0 = time.perf_counter()
    E_sdp = solve_pauli_relaxation(
        basis=basis.words,
        operator=H_dict_6,
        symmetry_manager=sym_none_6,
        sense="min",
        mosek_tol=1e-7,
    )
    dt = time.perf_counter() - t0
    gap = E_exact_6 - E_sdp
    rel = abs(gap / E_exact_6)
    print(f"  k={k:>2d}  {len(basis.words):>10d}  {E_sdp:>12.6f}  {gap:>12.6f}  {rel:>10.4%}  {dt:>10.3f}")

Heisenberg chain N=6, PBC
Exact ground-state energy: -2.802776

 Level  Basis size         E_SDP           Gap    Rel. gap    Time (s)
----------------------------------------------------------------------
  k= 1          19     -4.500000      1.697224    60.5551%       0.046
  k= 2         154     -2.802776     -0.000000     0.0000%       2.070


---
## 9. Symmetries: Tighter Bounds, Smaller SDPs

The Heisenberg Hamiltonian has rich symmetries that the `SymmetryManager` can exploit:

| Symmetry | Effect |
|----------|--------|
| **Sign** | Zeroes out moments with odd parity → fewer variables |
| **Translation** | Groups rotationally equivalent Pauli words → fewer variables |
| **Mirror** | Identifies spatially reflected words → fewer variables |
| **Permutation** | Identifies words related by $X \leftrightarrow Y \leftrightarrow Z$ relabelling |
| **Rotation (block diag.)** | Splits the moment matrix into smaller blocks by signature |
| **Real basis ($\tilde{Y}=iY$)** | Makes the moment matrix real-symmetric (no information loss) |

Using all symmetries simultaneously gives the tightest bound with the smallest SDP. Note that symmetries are **model specific**, so you should carefully analyse your model before applying each symmetry, as they might invalidate results if not physically valid.

In [31]:
# Compare: no symmetries vs. full symmetries
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
E_exact_6 = exact_energies[6]

configs = {
    "No symmetries": SymmetryManager.none(N),
    "Full symmetries": SymmetryManager.default_for_heisenberg(N),
}

basis_lv1 = generate_npa_basis(N, k=1)

print(f"N={N}, NPA level 1 ({len(basis_lv1.words)} basis words)")
print(f"Exact E0 = {E_exact_6:.6f}\n")

for label, sym in configs.items():
    t0 = time.perf_counter()
    E_sdp = solve_pauli_relaxation(
        basis=basis_lv1.words,
        operator=H_dict_6,
        symmetry_manager=sym,
        sense="min",
        mosek_tol=1e-7,
    )
    dt = time.perf_counter() - t0
    gap = E_exact_6 - E_sdp
    
    # Count variables
    reps, _ = build_block_reps(basis_lv1.words, sym)
    n_vars = len(reps[0].labels)
    n_blocks = len(reps)
    block_sizes = [r.label_idx.shape[0] for r in reps]

    print(f"  {label}:")
    print(f"    E_SDP = {E_sdp:.6f} (gap = {gap:.6f})")
    print(f"    Variables: {n_vars},  Blocks: {n_blocks},  Block sizes: {block_sizes}")
    print(f"    Time: {dt:.3f}s\n")

N=6, NPA level 1 (19 basis words)
Exact E0 = -2.802776

  No symmetries:
    E_SDP = -4.500000 (gap = 1.697224)
    Variables: 154,  Blocks: 1,  Block sizes: [19]
    Time: 0.036s

  Full symmetries:
    E_SDP = -4.500000 (gap = 1.697224)
    Variables: 7,  Blocks: 4,  Block sizes: [1, 6, 6, 6]
    Time: 0.013s



In [32]:
# Show how symmetries reduce variables across NPA levels
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
sym_full = SymmetryManager.default_for_heisenberg(N)

print(f"Variable reduction with full symmetries (N={N}):\n")
print(f"{'NPA Level':>10s}  {'Basis words':>12s}  {'Moment vars':>12s}")
print("-" * 40)

for k in [1, 2]:
    basis = generate_npa_basis(N, k=k)
    reps, _ = build_block_reps(basis.words, sym_full)
    n_vars = len(reps[0].labels)
    print(f"  k={k:>6d}  {len(basis.words):>12d}  {n_vars:>12d}")

Variable reduction with full symmetries (N=6):

 NPA Level   Basis words   Moment vars
----------------------------------------
  k=     1            19             7
  k=     2           154            39


---
## 10. Basis-Selection Optimisation

The full NPA level-2 basis can be large. The **optimisation layer** asks: *can we get a tight bound using only $k$ extra words from level 2, added to the full level-1 basis?*

`optimize_ground_energy()` is the main entry point. It:
1. Builds the **starting set** (level-1 words) and **adding set** (level-2 extras).
2. Evaluates each candidate selection by solving the SDP.
3. Uses a chosen optimiser (SA, PT, BO, RBM, or random) to find the best $k$ words from the adding set.

In [33]:
# Look at the basis sets
N = 6
starting_set, adding_set, final_set = build_npa_basis_sets(N, start_level=1, end_level=2)

print(f"N={N}, levels 1→2")
print(f"  Starting set (level ≤ 1): {len(starting_set)} words")
print(f"  Adding set   (level 2):   {len(adding_set)} words")
print(f"  Final set    (all):       {len(final_set)} words")

print(f"\nFirst 10 adding-set words:")
for i, w in enumerate(adding_set[:10]):
    print(f"  [{i}] {w}")

N=6, levels 1→2
  Starting set (level ≤ 1): 19 words
  Adding set   (level 2):   135 words
  Final set    (all):       154 words

First 10 adding-set words:
  [0] Z0Z1
  [1] Z0Z2
  [2] Z1Z2
  [3] Z0Z3
  [4] Z1Z3
  [5] Z2Z3
  [6] Z0Z4
  [7] Z1Z4
  [8] Z2Z4
  [9] Z3Z4


In [34]:
# Run a single optimisation: pick k=3 words from the adding set
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
sym_full = SymmetryManager.default_for_heisenberg(N)

result = optimize_ground_energy(
    N=N,
    hamiltonian_dict=H_dict_6,
    symmetry_manager=sym_full,
    start_level=1,
    end_level=2,
    k=3,
    method="sa",
    seed=42,
    verbose=True,
)

print(f"\n--- Result ---")
print(f"  Best SDP lower bound: {result.best_value:.6f}")
print(f"  Exact E0:             {exact_energies[N]:.6f}")
print(f"  Gap:                  {exact_energies[N] - result.best_value:.6f}")
print(f"  Selected indices:     {result.best_indices}")
print(f"  Time:                 {result.elapsed_s:.3f}s")
print(f"  Objective evaluations: {result.n_obj_evals}")

Spin optimisation: N=6
  Starting set: 19 words, Adding set: 135 words, k=3
  Method: sa
  Result: SDP lower bound = -2.911005, time = 1.19s, evals = 101

--- Result ---
  Best SDP lower bound: -2.911005
  Exact E0:             -2.802776
  Gap:                  0.108229
  Selected indices:     [43, 44, 61]
  Time:                 1.194s
  Objective evaluations: 101


---
## 11. Sweeping over *k* Values and Seeds

`sweep_k_values()` runs the optimisation for multiple $(k, \text{seed})$ pairs.

Key features:
- **`feedback=True`**: chains results across $k$ values — the best mask at $k_i$ is passed as warm-start to $k_{i+1}$.
- **`existing_results`**: skip already-computed jobs (for resume support).
- **`on_result`**: callback invoked after each new result (for atomic persistence).
- **`seeds`**: run multiple random seeds per $k$ for statistical robustness.

In [37]:
# Sweep over k=0,1,2,3 with 2 seeds
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
sym_full = SymmetryManager.default_for_heisenberg(N)

results = sweep_k_values(
    N=N,
    operator=H_dict_6,
    symmetry_manager=sym_full,
    start_level=1,
    end_level=2,
    k_values=[0, 1, 2, 3],
    seeds=[42, 43],
    method="sa",
    verbose=True,
    mosek_tol=1e-12,
)

# Display results
E0 = exact_energies[N]
print(f"\nExact E0 = {E0:.6f}")
print(f"\n{'k':>3s}  {'seed':>4s}  {'E_SDP':>12s}  {'gap':>10s}  {'time (s)':>8s}")
print("-" * 45)
for (k, seed), res in sorted(results.items()):
    gap = E0 - res.best_value
    print(f"{k:>3d}  {seed:>4d}  {res.best_value:>12.6f}  {gap:>10.6f}  {res.elapsed_s:>8.3f}")

Spin optimisation sweep: 100%|██████████| 8/8 [00:07<00:00,  1.09it/s, k=3, seed=43]


Best SDP lower bound per k:
  k=0: -4.500000
  k=1: -3.018578
  k=2: -2.931827
  k=3: -2.858169

Exact E0 = -2.802776

  k  seed         E_SDP         gap  time (s)
---------------------------------------------
  0    42     -4.500000    1.697224     0.016
  0    43     -4.500000    1.697224     0.013
  1    42     -3.018578    0.215803     1.200
  1    43     -3.018578    0.215803     1.170
  2    42     -2.931827    0.129052     1.213
  2    43     -2.931827    0.129052     1.218
  3    42     -2.897223    0.094448     1.236
  3    43     -2.858169    0.055394     1.240


---
## 12. Atomic Persistence with a Callback

The `on_result` callback is called after **every newly completed** `(k, seed)` job. This lets you save results **atomically** so that a crash loses at most the current computation.

The pattern:
1. Create a `RunDir` via `ArtifactManager.create_run()`.
2. Define a callback that appends the new result and calls `run.save_table()`.
3. Pass the callback as `on_result=` to `sweep_k_values()`.

In [38]:
# Set up a clean demo directory
demo_root = Path("spins_demo_results")
if demo_root.exists():
    shutil.rmtree(demo_root)
demo_root.mkdir()

am = ArtifactManager(demo_root)

# Create a run
N = 6
sweep_config = {
    "model": "heisenberg",
    "N": N,
    "boundary": "periodic",
    "start_level": 1,
    "end_level": 2,
    "method": "sa",
}

run = am.create_run(
    artifact="spin_optimization_sweep",
    name="heisenberg_N6_sa_demo",
    config=sweep_config,
)
print(f"Run directory: {run.path}")
print(f"Config hash:   {config_hash(sweep_config)}")

Run directory: spins_demo_results/spin_optimization_sweep/v1/heisenberg_N6_sa_demo
Config hash:   22416bce358f22a0


In [39]:
# Define the atomic callback
_acc_k = []
_acc_seed = []
_acc_best_value = []
_acc_elapsed_s = []
_acc_n_obj_evals = []
_acc_mask_bits = []

def on_result(k: int, seed: int, res: SpinOptimizationResult) -> None:
    """Save every new result atomically via RunDir.save_table()."""
    packed_mask = np.packbits(res.mask.astype(np.uint8))

    _acc_k.append(k)
    _acc_seed.append(seed)
    _acc_best_value.append(res.best_value)
    _acc_elapsed_s.append(res.elapsed_s)
    _acc_n_obj_evals.append(res.n_obj_evals)
    _acc_mask_bits.append(packed_mask)

    run.save_table(
        k=np.array(_acc_k, dtype=np.int32),
        seed=np.array(_acc_seed, dtype=np.int32),
        best_value=np.array(_acc_best_value, dtype=np.float64),
        elapsed_s=np.array(_acc_elapsed_s, dtype=np.float64),
        n_obj_evals=np.array(_acc_n_obj_evals, dtype=np.int32),
        mask_bits=np.stack(_acc_mask_bits, axis=0),
    )
    print(f"    Checkpoint: saved {len(_acc_k)} result(s) to disk")

print("Callback defined ✓")

Callback defined ✓


In [40]:
# Run the sweep with atomic persistence
N = 6
H_dict_6 = heisenberg_hamiltonian_dict(N, boundary="periodic")
sym_full = SymmetryManager.default_for_heisenberg(N)

sweep_results = sweep_k_values(
    N=N,
    operator=H_dict_6,
    symmetry_manager=sym_full,
    start_level=1,
    end_level=2,
    k_values=[0, 1, 2],
    seeds=[42, 43],
    method="sa",
    on_result=on_result,
    verbose=True,
)

# Finalise metadata
run.update_meta(
    status="complete",
    k_values=[0, 1, 2],
    seeds=[42, 43],
    total_runs=len(_acc_k),
)

print(f"\nSweep complete — {len(_acc_k)} results saved to {run.path}")

Spin optimisation sweep:  33%|███▎      | 2/6 [00:00<00:00, 57.42it/s, k=1, seed=42]

    Checkpoint: saved 1 result(s) to disk
    Checkpoint: saved 2 result(s) to disk


Spin optimisation sweep:  50%|█████     | 3/6 [00:01<00:01,  2.60it/s, k=1, seed=43]

    Checkpoint: saved 3 result(s) to disk


Spin optimisation sweep:  67%|██████▋   | 4/6 [00:02<00:01,  1.61it/s, k=2, seed=42]

    Checkpoint: saved 4 result(s) to disk


Spin optimisation sweep:  83%|████████▎ | 5/6 [00:03<00:00,  1.27it/s, k=2, seed=43]

    Checkpoint: saved 5 result(s) to disk


Spin optimisation sweep: 100%|██████████| 6/6 [00:04<00:00,  1.31it/s, k=2, seed=43]

    Checkpoint: saved 6 result(s) to disk

Best SDP lower bound per k:
  k=0: -4.500000
  k=1: -3.018578
  k=2: -2.858169

Sweep complete — 6 results saved to spins_demo_results/spin_optimization_sweep/v1/heisenberg_N6_sa_demo


---
## 13. Loading and Discovering Saved Results

You can come back later and reload everything from disk.

In [41]:
import os

# What's on disk?
print("On-disk layout:\n")
for dirpath, dirnames, filenames in os.walk(demo_root):
    level = dirpath.replace(str(demo_root), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    sub_indent = "  " * (level + 1)
    for f in filenames:
        fpath = Path(dirpath) / f
        size = fpath.stat().st_size
        print(f"{sub_indent}{f}  ({size} bytes)")

On-disk layout:

spins_demo_results/
  spin_optimization_sweep/
    v1/
      heisenberg_N6_sa_demo/
        meta.json  (452 bytes)
        data.npz  (1772 bytes)


In [42]:
# Re-open and load
am2 = ArtifactManager(demo_root)

print("Available artifacts:")
for artifact_dir in sorted(am2.root.iterdir()):
    if artifact_dir.is_dir():
        runs = am2.list_runs(artifact_dir.name)
        print(f"  {artifact_dir.name}: {len(runs)} run(s)")

# Load the table
loaded_run = am2.open_run(
    artifact="spin_optimization_sweep",
    name="heisenberg_N6_sa_demo",
)
data = loaded_run.load_table()

print(f"\nLoaded columns: {sorted(data.keys())}")
print(f"Number of rows: {len(data['k'])}")
print(f"\n{'k':>3s}  {'seed':>4s}  {'best_value':>12s}  {'elapsed_s':>9s}")
print("-" * 35)
for i in range(len(data["k"])):
    print(f"{data['k'][i]:>3d}  {data['seed'][i]:>4d}  "
          f"{data['best_value'][i]:>12.6f}  {data['elapsed_s'][i]:>9.3f}")

Available artifacts:
  spin_optimization_sweep: 1 run(s)

Loaded columns: ['best_value', 'elapsed_s', 'k', 'mask_bits', 'n_obj_evals', 'seed']
Number of rows: 6

  k  seed    best_value  elapsed_s
-----------------------------------
  0    42     -4.500000      0.016
  0    43     -4.500000      0.013
  1    42     -3.018578      1.114
  1    43     -3.018578      1.115
  2    42     -2.931827      1.154
  2    43     -2.858169      1.154


---
## Summary

In this tutorial we covered the full spin-chain pipeline:

1. **Exact diagonalisation** with QuTiP for small system sizes.
2. **Pauli-word algebra** - the symbolic backbone of the NPA hierarchy.
3. **NPA basis generation** - systematic and paper-specific bases.
4. **Moment-matrix SDP relaxation** - certified lower bounds on $E_0$.
5. **Symmetries** - sign, translation, mirror, permutation, block diagonalisation, and the real-basis trick.
6. **Basis-selection optimisation** - picking the best $k$ extra words.
7. **Atomic persistence** - crash-safe checkpointing with the `ArtifactManager`.

**Next:** see the companion notebook *General Observable Bounds* for bounding arbitrary observables beyond ground-state energy.